## Required libraries

In [45]:
import re                                                           # Regular expressions library for string operations
import time                                                         # Time-related functions
from datetime import datetime                                       # Date and time functionality combined
import os                                                           # Interact with the operating system
from collections import defaultdict                                 # Dictionary subclass that provides a default value for non-existent keys

import openai                                                       # OpenAI's Python client library

from concurrent.futures import ThreadPoolExecutor, as_completed     # Parallel processing capabilities

import nltk                                                         # Platform for building programs to work with human language data
import spacy

from nltk.corpus import stopwords, words                            # Remove common words, access words corpus
from nltk.stem import WordNetLemmatizer                             # Reduce words to their root form
import language_tool_python                                         # Python wrapper for LanguageTool for grammar and spelling checking

from language_tool_python import LanguageTool

import pandas as pd                                                 # Data manipulation and analysis library

## Environment Configuration & Set Up

In [46]:
nlp = spacy.load("en_core_web_sm")
tool = LanguageTool('en-US')

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('words', quiet=True)

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

english_words = set(words.words())
stopwords_list = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

tool = language_tool_python.LanguageTool('en-US')

executor = ThreadPoolExecutor(max_workers=10)

open_ai_keys = [
    os.getenv("OPENAI_KEY_1"),
    os.getenv("OPENAI_KEY_2"),
    os.getenv("OPENAI_KEY_3"),
    os.getenv("OPENAI_KEY_4"),
    os.getenv("OPENAI_KEY_5"),
    os.getenv("OPENAI_KEY_6"),
    os.getenv("OPENAI_KEY_7"),
    os.getenv("OPENAI_KEY_8"),
    os.getenv("OPENAI_KEY_9"),
    os.getenv("OPENAI_KEY_10"),
    os.getenv("OPENAI_KEY_11"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_13"),
    os.getenv("OPENAI_KEY_14"),
    os.getenv("OPENAI_KEY_15"),
    os.getenv("OPENAI_KEY_16"),
    os.getenv("OPENAI_KEY_17"),
    os.getenv("OPENAI_KEY_18"),
    os.getenv("OPENAI_KEY_19"),
    os.getenv("OPENAI_KEY_20")
]

openai.api_key = open_ai_keys[0]

## Loading Debate Transcript


In [26]:
transcript = "2020/transcripts/debate-1/transcript-2020-9-29.txt"

## Transcript Pre-processing & Inspection

We will process the debate sentence by sentence.

In [27]:
def sentence_segmentation(transcript):
    with open(transcript, 'r') as file:
        data = file.read().replace('\n', ' ')

    doc = nlp(data)
    sentences = [sent.text.strip() for sent in doc.sents]
    return sentences

Overview of the transcript and speakers

In [28]:
def inspect(filename):
    with open(filename, 'r') as file:
        data = file.read().replace(". . .", "<ellipsis>")
    segments = re.split(r'([A-Z]+:)', data)
    speaker_stats = defaultdict(lambda: defaultdict(int))
    total_stats = {'total_sentences': 0, 'total_paragraphs': 0, 'total_words': 0}
    current_speaker = None
    for segment in segments:
        if re.match(r'[A-Z]+:', segment):
            current_speaker = segment[:-1]
            speaker_stats[current_speaker]['paragraphs'] += 1
            total_stats['total_paragraphs'] += 1
        elif current_speaker is not None:
            words = re.findall(r'\b\w+\b', segment)
            sentences = nltk.sent_tokenize(segment)
            sentences = [sentence.replace("<ellipsis>", ". . .") for sentence in sentences]
            speaker_stats[current_speaker]['words'] += len(words)
            speaker_stats[current_speaker]['sentences'] += len(sentences)
            total_stats['total_words'] += len(words)
            total_stats['total_sentences'] += len(sentences)
            
    inspection = {
        'number_of_speakers': len(speaker_stats),
        'speaker_stats': dict(speaker_stats),
        'total_stats': total_stats,
    }
    
    speaker_count = inspection['number_of_speakers']
    overall_inspection_df = pd.Series(inspection['total_stats'])
    speaker_inspection_df = pd.DataFrame(inspection["speaker_stats"])
    
    display("Total Unique Speakers: " + str(speaker_count), overall_inspection_df, speaker_inspection_df)

In [29]:
inspect(transcript)

'Total Unique Speakers: 3'

total_sentences      2021
total_paragraphs      856
total_words         19948
dtype: int64

,WALLACE,BIDEN,TRUMP
paragraphs,246,269,341
words,4899,7136,7913
sentences,476,699,846


In [18]:
sentence_list = sentence_segmentation(transcript)

## `generate_scoreboard`

In [32]:
def generate_empty_scoreboard(filename):
    with open(filename, 'r') as file:
        data = file.read().replace(". . .", "<ellipsis>")
        
    segments = re.split(r'([A-Z]+:)', data)
    sentences_list = []
    current_speaker = None
    all_sentences = []  # List to hold all sentences

    for segment in segments:
        if re.match(r'[A-Z]+:', segment):
            current_speaker = segment[:-1]
        elif current_speaker:
            # Segment the text into sentences
            doc = nlp(segment.strip())
            sentence_list = [sent.text for sent in doc.sents]
            
            for sentence in sentence_list:
                sentence = sentence.replace("<ellipsis>", ". . .")
                sentences_list.append({'Speaker': current_speaker, 'Sentence': sentence})
                all_sentences.append(sentence)  # Append the sentence to the list
                
    empty_scoreboard = pd.DataFrame(sentences_list)
    empty_scoreboard.insert(2, "Sentence_Score", 0)
    empty_scoreboard.insert(3, "Cumulative_Speaker_Score", 0)
    
    return empty_scoreboard, all_sentences

In [34]:
empty_scoreboard, sentence_list = generate_empty_scoreboard(transcript)

display(empty_scoreboard)

,Speaker,Sentence,Sentence_Score,Cumulative_Speaker_Score
0,WALLACE,Good evening from the Health Education Campus of Case Western Reserve University and the Cleveland Clinic.,0,0
1,WALLACE,I’m Chris Wallace of Fox News and I welcome you to the first of the 2020 presidential debates between President Donald J. Trump and former Vice President Joe Biden.,0,0
2,WALLACE,This debate is sponsored by the Commission on Presidential Debates.,0,0
3,WALLACE,"The Commission has designed the format, six roughly 15-minute segments with two-minute answers from each candidate to the first question, then open discussion for the rest of each segment.",0,0
4,WALLACE,Both campaigns have agreed to these rules.,0,0
...,...,...,...,...
2084,WALLACE,We want to thank Case Western Reserve University and the Cleveland Clinic for hosting this event.,0,0
2085,WALLACE,"The next debate, sponsored by the Commission on Presidential Debates, will be one week from tomorrow, October 7th, at the University of Utah in Salt Lake City.",0,0
2086,WALLACE,"The two Vice-Presidential nominees, Vice President Mike Pence and Senator Kamala Harris will debate at 9:00 PM Eastern that night.",0,0
2087,WALLACE,We hope you watch.,0,0


## `base_calc`

In [36]:
def base_calc(sentence):
    doc = nlp(sentence)
    base_score = 0

    base_score += len(doc)
    base_score += sum(1 for token in doc if not token.is_stop)
    base_score += sum(1 for token in doc if token.text in english_words)
    base_score += sum(1 for token in doc if token.lemma_ in english_words)
    correct_punctuation = sum(1 for token in doc if token.is_punct)
    base_score += correct_punctuation
    if sentence.count('.') + sentence.count('?') + sentence.count('!') == 1:
        base_score += 10

    return base_score

## `EPL_lookup`

Ethos is related to the speaker's character, reputation, or credibility. Ethos can be conveyed through tone and style of the message, as well as through the way the speaker or writer refers to differing views. It can also be affected by the speaker's reputation as it exists independently from the message–his or her expertise in the field, his or her previous record or integrity, etc.

Pathos relates to the emotional impact of the speech or text on the audience. It often invokes feelings that already reside in them. It can include certain story elements, such as setting, character, and plot. It can also use language that generates strong emotions, and rhetorical devices such as metaphors, similes, and analogies.

Logos relates to the logical and rational appeal of the argument. It includes the internal consistency of the message–the clarity of the claim, the logic of its reasons, and the effectiveness of its supporting evidence. It also includes factual data, statistics, research, and reasoning.

In [47]:
epl = {
    'E': {
        'Credibility':0.000,
        'Trust': 0,
        'Authority': 0,
        'Legitimacy': 0,
        'Reliability': 0,
        'Honesty': 0,
        'Expertise': 0,
        'Respect': 0
    },
    'P': {
        'Joy': 0,
        'Anger': 0,
        'Trust': 0,
        'Surprise': 0,
        'Fear': 0,
        'Disgust': 0,
        'Sadness': 0,
        'Happiness': 0,
        'Love': 0,
        'Anticipation': 0
    },
    'L': {
        'Evidence': 0,
        'Scientific': 0,
        'Rational': 0,
        'Logical': 0,
        'Reason': 0,
        'Proof': 0,
        'Facts': 0,
        'Analysis': 0,
        'Data': 0,
        'Consistency': 0
    }
}

In [39]:
from epl import e, p, l

def lemmatize_dict(d):
    return {k: [nlp(word)[0].lemma_ for word in v] for k, v in d.items()}

e = lemmatize_dict(e)
p = lemmatize_dict(p)
l = lemmatize_dict(l)

In [ ]:
def prompt_epl_score(lemma):
    prompt = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {
                "role": "system",
                "content": "You are a highly skilled AI developed by OpenAI. Your task is to assess words based on their contribution to ethos, pathos, and logos in a debate context."
            },
            {
                "role": "user",
                "content": f"Consider the word '{lemma}'. Rate its typical contribution to each subcategory within ethos, pathos, and logos when used appropriately in a debate. Use a scale of 0 (the word does not contribute at all) to 5 (the word significantly contributes). Provide your rating in a dictionary format as follows: 'E': {{'Credibility': score, 'Trust': score, 'Authority': score, 'Legitimacy': score, 'Reliability': score, 'Honesty': score, 'Expertise': score, 'Respect': score}}, 'P': {{'Joy': score, 'Anger': score, 'Trust': score, 'Surprise': score, 'Fear': score, 'Disgust': score, 'Sadness': score, 'Happiness': score, 'Love': score, 'Anticipation': score}}, 'L': {{'Evidence': score, 'Scientific': score, 'Rational': score, 'Logical': score, 'Reason': score, 'Proof': score, 'Facts': score, 'Analysis': score, 'Data': score, 'Consistency': score}}. Remember, your rating should reflect a general usage of the word, and actual contribution can vary depending on the context."
            }
        ]
    )
    return prompt


In [37]:
def ethos_pathos_logos_calc(sentence):
    doc = nlp(sentence)
    ethos_score, pathos_score, logos_score = 0, 0, 0

    # Ethos score
    for key, values in e.items():
        ethos_score += sum(int(key) for token in doc if token.lemma_ in values)
        
    # Pathos score
    for key, values in p.items():
        pathos_score += sum(int(key) for token in doc if token.lemma_ in values)

    # Logos score
    for key, values in l.items():
        logos_score += sum(int(key) for token in doc if token.lemma_ in values)

    return ethos_score, pathos_score, logos_score

## `apply_score`

In [40]:
def apply_score(df):
    df['Sentence_Score'] = df['Sentence'].apply(base_calc)
    df['Ethos_Bonus'], df['Pathos_Bonus'], df['Logos_Bonus'] = zip(*df['Sentence'].apply(ethos_pathos_logos_calc))
    df['Cumulative_Speaker_Score'] = df.groupby('Speaker')['Sentence_Score'].cumsum()
    df['Total_Bonus'] = df['Ethos_Bonus'] + df['Pathos_Bonus'] + df['Logos_Bonus']
    
    return df

In [41]:
scoreboard = apply_score(empty_scoreboard)

In [ ]:
scoreboard

In [ ]:
view = scoreboard.sort_values(["Pathos_Bonus"], ascending=False)

view

In [ ]:
next_sentence = ""

def prompt_standard(sentence):
    prompt = openai.ChatCompletion.create(
                    model="gpt-4",
                    messages=[
                        {"role": "system", "content": "You are a master debater and assistant to the judge of the US Presidential Debates."},
                        {"role": "user", "content": f"Analyze this phrase (may be incomplete) for Ethos, Pathos, and Logos from 0.0 - 9.9: '{sentence}'. Return a compact dictionary format like this: 'Ethos: x.x, Pathos: y.y, Logos: z.z, Ethos_Explanation: text, Pathos_Explanation: text, Logos_Explanation: text'. Explanations must be exactly 1 sentence long."}
                    ]
                )
    return prompt

def prompt_special(sentence, previous_sentence,):
    prompt = openai.ChatCompletion.create(
                    model="gpt-4",
                    messages=[
                        {"role": "system", "content": "You are a master debater and assistant to the judge of the US Presidential Debates."},
                        {"role": "user", "content": f"Analyze this question/exclamation phrase for Ethos, Pathos, and Logos from 0.0 - 9.9: '{sentence}', given that the statement it is responding to: {previous_sentence}. Return a compact dictionary format like this: 'Ethos: x.x, Pathos: y.y, Logos: z.z, Ethos_Explanation: text, Pathos_Explanation: text, Logos_Explanation: text'. Explanations must be exactly 1 sentence long."}
                    ]
                )
    return prompt

def prompt_interruption(sentence, previous_sentence, next_sentence):
    prompt = openai.ChatCompletion.create(
                    model="gpt-4",
                    messages=[
                        {"role": "system", "content": "You are a master debater and assistant to the judge of the US Presidential Debates."},
                        {"role": "user", "content": f"Analyze this interrupted phrase (may be incomplete) for Ethos, Pathos, and Logos from 0.0 - 9.9: '{sentence}', given that the statement was responding to: {previous_sentence}; but interrupted by {next_sentence}. Return a compact dictionary format like this: 'Ethos: x.x, Pathos: y.y, Logos: z.z, Ethos_Explanation: text, Pathos_Explanation: text, Logos_Explanation: text'. Explanations must be exactly 1 sentence long."}
                    ]
                )
    return prompt

def prompt_stats(sentence):
    prompt = openai.ChatCompletion.create(
                    model="gpt-4",
                    messages=[
                        {"role": "system", "content": "You are assisting in a debate analysis."},
                        {"role": "user", "content": f"Does this sentence contain any statistics (S)? '{sentence}' If so, identify it. Provide the first and last 2 words of the quote for the statistic. Return a compact dictionary in the format: 'S: 0/1, S_Quote_Start: \"start\", S_Quote_End: \"end\""}
                    ]
                )
    return prompt



In [ ]:
speaker_list = scoreboard["Speaker"].tolist()
sentence_list = scoreboard["Sentence"].tolist()

def worker(task, key, sentence_list, max_retries=3):
    
    global total_requests, total_errors
    
    openai.api_key = key
    total_requests += max_retries
    i, sentence = task
    response_dict = {"EPL": "", "Stats": ""}
    previous_sentence = sentence_list[i - 1] if i > 0 else ''
    
    # Analyze for statistic or claim if sentence is longer than 9 words
    if len(nltk.word_tokenize(sentence)) > 9:
        for _ in range(max_retries):
            try:
                stat_claim_response = prompt_stats(sentence)
                response_dict["Stat_Claim"] = stat_claim_response['choices'][0]['message']['content']
                
                with open("stats.txt", "a") as stat_claim_file:
                    stat_claim_file.write(f"Sentence {i}: {sentence}\nResult: {response_dict['Stat_Claim']}\n")

                break

            except Exception as e:
                total_errors += 1
                if _ == max_retries - 1:
                    raise e
                time.sleep(1)

    for _ in range(max_retries):
        try:
            if "crosstalk" in sentence or sentence.strip().endswith("-"):
                response = prompt_interruption(sentence, previous_sentence)

            elif sentence.strip().endswith("?") or sentence.strip().endswith("!"):
                response = prompt_special(sentence, previous_sentence)
                
            else:
                response = prompt_standard(sentence)
                
            response_dict["EPL"] = response['choices'][0]['message']['content']
            
            with open("EPL.txt", "a") as epl_file:
                epl_file.write(f"Sentence {i}: {sentence}\nResult: {response_dict['EPL']}\n")

            break
        
        except Exception as e:
            total_errors += 1
            if _ == max_retries - 1:
                raise e
            time.sleep(1)

    return i, response_dict["EPL"], response_dict["Stats"]

In [ ]:
def analyze(sentence_list, keys):
    
    global total_requests, total_errors, total_tasks, completed_tasks
    total_requests = total_errors = completed_tasks = 0
    total_tasks = len(sentence_list) - 1
    
    tasks = [(i, sentence) for i, sentence in enumerate(sentence_list)]

    futures = []
    
    with ThreadPoolExecutor() as executor:
        for i in range(0, len(tasks), 10):
            key = keys[i // 10 % len(keys)]
            futures.extend(executor.submit(worker, task, key, sentence_list) for task in tasks[i:i+10])

        for future in as_completed(futures):
            try:
                index, epl_result, stat_claim_result = future.result()
                completed_tasks += 1
                with open("log.txt", "a") as log_file:
                    log_file.write(f"Task {completed_tasks} of {total_tasks} completed at index {index} on {datetime.now()}\n")
            except Exception as e:
                with open("log.txt", "a") as log_file:
                    log_file.write(f"An error occurred at {datetime.now()}: {e}\n")


    with open("log.txt", "a") as log_file:
        log_file.write(f"Total requests: {total_requests}.\nTotal errors: {total_errors}.\nTotal tasks: {total_tasks}.\nCompleted tasks: {completed_tasks}.\n")

    return futures

In [ ]:
analyze(sentence_list, open_ai_keys)

In [ ]:
def parse_output_to_dataframe(filename):
    data = {"Sentence": [], "Speaker": [], "Ethos": [], "Pathos": [], "Logos": [], 
            "Ethos_Explanation": [], "Pathos_Explanation": [], "Logos_Explanation": []}
    current_sentence, current_speaker = None, None

    with open(filename, "r") as f:
        for line in f:
            if line.startswith("Sentence"):
                sentence_match = re.search(r"Sentence (\d+)", line)
                speaker_match = re.search(r"\((.*?)\):", line)
                
                if sentence_match:
                    current_sentence = int(sentence_match.group(1))
                    
                if speaker_match:
                    current_speaker = speaker_match.group(1)
                    
            elif line.startswith("Result"):
                ethos_match = re.search(r"Ethos: (\d+\.?\d*),", line)
                pathos_match = re.search(r"Pathos: (\d+\.?\d*),", line)
                logos_match = re.search(r"Logos: (\d+\.?\d*),", line)
                ethos_explanation_match = re.search(r"Ethos_Explanation: (.*?),", line)
                pathos_explanation_match = re.search(r"Pathos_Explanation: (.*?),", line)
                logos_explanation_match = re.search(r"Logos_Explanation: (.*?)(?=Sentence|$)", line)

                data["Sentence"].append(current_sentence)
                data["Speaker"].append(current_speaker)
                data["Ethos"].append(float(ethos_match.group(1)) if ethos_match else None)
                data["Pathos"].append(float(pathos_match.group(1)) if pathos_match else None)
                data["Logos"].append(float(logos_match.group(1)) if logos_match else None)
                data["Ethos_Explanation"].append(ethos_explanation_match.group(1) if ethos_explanation_match else None)
                data["Pathos_Explanation"].append(pathos_explanation_match.group(1) if pathos_explanation_match else None)
                data["Logos_Explanation"].append(logos_explanation_match.group(1) if logos_explanation_match else None)

    df = pd.DataFrame(data)
    
    df.sort_values(by=['Sentence'], inplace=True)
    df2 = df.set_index("Sentence", drop=True)
    
    df2["Speaker"] = scoreboard["Speaker"]
    
    df2.insert(1, "Sentence", scoreboard["Sentence"])
    df2.insert(8, "Sentence_Score", scoreboard["Sentence_Score"])
    df2.insert(9, "Cumulative_Speaker_Score", scoreboard["Cumulative_Speaker_Score"])

    return df2

In [ ]:
filled_scoreboard = parse_output_to_dataframe("output.txt")
filled_scoreboard

In [ ]:
def save_scoreboard_as_csv(df):
    now = datetime.now()
    date = [str(now.strftime("%d/%m/%Y %H:%M:%S")).split(" ")][0][0].replace("/", "").replace("0206", "6-02-")
    t = time.localtime()
    current_time = time.strftime("%H:%M", t).replace(":","")
    path = "2020/saved/csv"
    filename = "/2020_debate-1_EPL_scoreboard_" + str(date) + "-" + str(current_time) + ".csv"
    namepath = path + filename
    print("File saved as: ", filename, " in ", path)
    df.to_csv(namepath, index=False)

In [ ]:
def save_scoreboard_as_json(df):
    now = datetime.now()
    date = [str(now.strftime("%d/%m/%Y %H:%M:%S")).split(" ")][0][0].replace("/", "").replace("0206", "6-02-")
    t = time.localtime()
    current_time = time.strftime("%H:%M", t).replace(":","")
    path = "2020/saved/json"
    filename = "/2020_debate-1_EPL_scoreboard_" + str(date) + "-" + str(current_time) + ".json"
    namepath = path + filename
    print("File saved as: ", str(filename), " in ", str(path))
    df.to_json(namepath, indent=4, orient="index")

In [ ]:
def multisave(df):
    save_scoreboard_as_csv(df)
    save_scoreboard_as_json(df)

In [ ]:
multisave(filled_scoreboard)